# ⚡ Smart Energy Consumption Predictor & Wastage Detection System


## Install the required Libraries

In [1]:
!pip install streamlit xgboost plotly pyngrok -q
print('All libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.0 MB/s eta 0:00:00
All libraries installed!


## Generating the  Dataset

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

np.random.seed(42)
dates = pd.date_range(start='2024-01-01', periods=200, freq='h')
n = 200

hour = dates.hour
month = dates.month
day_of_week = dates.dayofweek
is_weekend = (day_of_week >= 5).astype(int)

temperature = 22 + 8 * np.sin(2 * np.pi * hour / 24) + np.random.normal(0, 2, n)
occupancy = np.where((is_weekend == 0) & (hour >= 8) & (hour <= 18),
                     np.random.uniform(60, 100, n),
                     np.random.uniform(5, 30, n))
consumption = (
    40 + 20 * np.sin(2 * np.pi * hour / 24)
    + 0.4 * temperature + 0.2 * occupancy
    + np.random.normal(0, 5, n)
)
consumption = np.clip(consumption, 10, 200)

df = pd.DataFrame({
    'timestamp': dates,
    'consumption_kwh': np.round(consumption, 2),
    'temperature_c': np.round(temperature, 1),
    'occupancy_pct': np.round(np.clip(occupancy, 0, 100), 1),
    'hour': hour,
    'day_of_week': day_of_week,
    'month': month,
    'is_weekend': is_weekend,
    'is_holiday': 0
})

df.to_csv('energy_data.csv', index=False)
print(f'Dataset ready: {len(df)} rows')
df.head(10)

Dataset ready: 200 rows


,timestamp,consumption_kwh,temperature_c,occupancy_pct,hour,day_of_week,month,is_weekend,is_holiday
0,2024-01-01 00:00:00,51.76,23.0,29.1,0,0,1,0,0
1,2024-01-01 01:00:00,55.13,23.8,14.4,1,0,1,0,0
2,2024-01-01 02:00:00,60.38,27.3,12.1,2,0,1,0,0
3,2024-01-01 03:00:00,67.45,30.7,26.7,3,0,1,0,0
4,2024-01-01 04:00:00,71.07,28.5,10.6,4,0,1,0,0
5,2024-01-01 05:00:00,72.68,29.3,29.1,5,0,1,0,0
6,2024-01-01 06:00:00,75.68,33.2,5.3,6,0,1,0,0
7,2024-01-01 07:00:00,77.42,31.3,29.2,7,0,1,0,0
8,2024-01-01 08:00:00,86.26,28.0,94.7,8,0,1,0,0
9,2024-01-01 09:00:00,80.41,28.7,96.5,9,0,1,0,0


 ## Training the ML Models

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

FEATURES = ['hour','day_of_week','month','is_weekend','is_holiday','temperature_c','occupancy_pct']
TARGET = 'consumption_kwh'

df2 = pd.read_csv('energy_data.csv', parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)
df2['lag_1h'] = df2[TARGET].shift(1)
df2['lag_24h'] = df2[TARGET].shift(24)
df2['rolling_mean_24h'] = df2[TARGET].shift(1).rolling(24).mean()
df2 = df2.dropna().reset_index(drop=True)

feature_cols = FEATURES + ['lag_1h', 'lag_24h', 'rolling_mean_24h']
X, y = df2[feature_cols], df2[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'MAE':     round(mean_absolute_error(y_true, y_pred), 3),
        'RMSE':    round(np.sqrt(mean_squared_error(y_true, y_pred)), 3),
        'R2':      round(r2_score(y_true, y_pred), 4),
        'MAPE(%)': round(np.mean(np.abs((y_true - y_pred)/(y_true+1e-9)))*100, 2)
    }

results = []

print('Training Linear Regression...')
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
results.append(evaluate('Linear Regression', y_test, y_pred_lr))
print('Done!')

print('Training Random Forest...')
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(evaluate('Random Forest', y_test, y_pred_rf))
print('Done!')

print('Training XGBoost...')
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                               max_depth=6, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
results.append(evaluate('XGBoost', y_test, y_pred_xgb))
print('Done!')

results_df = pd.DataFrame(results)
print('\n All Models Trained & Predictions Made!')
print('\n Model Performance:')
print(results_df.to_string(index=False))

Training Linear Regression...
Done!
Training Random Forest...
Done!
Training XGBoost...
Done!

 All Models Trained & Predictions Made!

 Model Performance:
            Model   MAE  RMSE     R2  MAPE(%)
Linear Regression 4.926 6.074 0.8703    10.85
    Random Forest 5.124 6.247 0.8628    10.56
          XGBoost 5.561 6.699 0.8423    11.42


## 🔍 Step 4 — Anomaly Detection

In [4]:
df_anom = df2.copy()
df_anom['expected'] = df_anom.groupby('hour')['consumption_kwh'].transform('mean')
df_anom['std']      = df_anom.groupby('hour')['consumption_kwh'].transform('std')
df_anom['z_score']  = (df_anom['consumption_kwh'] - df_anom['expected']) / (df_anom['std'] + 1e-9)
df_anom['detected_anomaly'] = (df_anom['z_score'].abs() > 2.5).astype(int)
df_anom['wastage_kwh'] = np.where(
    df_anom['detected_anomaly'] == 1,
    df_anom['consumption_kwh'] - df_anom['expected'], 0)
df_anom['wastage_kwh'] = df_anom['wastage_kwh'].clip(lower=0)

print('Wastage Detection Complete!')
print(f'Anomalies detected : {df_anom["detected_anomaly"].sum()}')
print(f'Total Wastage      : {df_anom["wastage_kwh"].sum():,.1f} kWh')
print(f'Est. Cost (₹6/kWh): ₹{df_anom["wastage_kwh"].sum()*6:,.0f}')

Wastage Detection Complete!
Anomalies detected : 0
Total Wastage      : 0.0 kWh
Est. Cost (₹6/kWh): ₹0


## 📊 Step 5 — Visualizations

In [5]:
import plotly.graph_objects as go
import plotly.express as px

# ── Chart 1: Actual vs Predicted ──────────────────────────────
test_times = df2.iloc[X_test.index]['timestamp'].values

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=test_times, y=y_test.values,
                           mode='lines', name='Actual',
                           line=dict(color='#ffffff', width=2)))
fig1.add_trace(go.Scatter(x=test_times, y=y_pred_lr,
                           mode='lines', name='Linear Regression',
                           line=dict(color='#7c83fd', width=1.5, dash='dot')))
fig1.add_trace(go.Scatter(x=test_times, y=y_pred_rf,
                           mode='lines', name='Random Forest',
                           line=dict(color='#64ffda', width=1.5, dash='dash')))
fig1.add_trace(go.Scatter(x=test_times, y=y_pred_xgb,
                           mode='lines', name='XGBoost',
                           line=dict(color='#ff6b9d', width=1.5, dash='dashdot')))
fig1.update_layout(title='Actual vs Predicted Energy Consumption',
                   xaxis_title='Time', yaxis_title='kWh', height=400)
fig1.show()

# ── Chart 2: Model Comparison ─────────────────────────────────
fig2 = px.bar(results_df, x='Model', y='R2', color='Model',
              title='Model R² Score (higher = better)',
              color_discrete_sequence=['#7c83fd','#64ffda','#ff6b9d'],
              text='R2')
fig2.update_traces(textposition='outside')
fig2.update_layout(height=350, showlegend=False)
fig2.show()

# ── Chart 3: Anomaly Detection ────────────────────────────────
normal = df_anom[df_anom['detected_anomaly'] == 0]
anomal = df_anom[df_anom['detected_anomaly'] == 1]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=normal['timestamp'], y=normal['consumption_kwh'],
                           mode='markers', name='Normal',
                           marker=dict(color='#64ffda', size=4)))
fig3.add_trace(go.Scatter(x=anomal['timestamp'], y=anomal['consumption_kwh'],
                           mode='markers', name='Wastage/Anomaly',
                           marker=dict(color='#ff4757', size=10, symbol='x')))
fig3.add_trace(go.Scatter(x=df_anom['timestamp'], y=df_anom['expected'],
                           mode='lines', name='Expected',
                           line=dict(color='#ffd700', width=1.5, dash='dot')))
fig3.update_layout(title='Wastage & Anomaly Detection',
                   xaxis_title='Time', yaxis_title='kWh', height=400)
fig3.show()

# ── Chart 4: Feature Importance ──────────────────────────────
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance')

fig4 = px.bar(feat_imp, x='Importance', y='Feature', orientation='h',
              title='Feature Importances (Random Forest)',
              color='Importance', color_continuous_scale='teal')
fig4.update_layout(height=400, coloraxis_showscale=False)
fig4.show()

print(' All charts displayed!')

 All charts displayed!


## 🚀 Step 6 — Launch Streamlit App (with Public URL)

In [6]:
# import subprocess, threading, time, os
# from pyngrok import ngrok

# # Kill old processes
# os.system("pkill -f streamlit")
# ngrok.kill()
# time.sleep(3)

# # Set token
# ngrok.set_auth_token("3Fa5vHo8XqriknbjjVZu0XKpsHz_4aEPpnZu56jrL8EtYWbuQ")

# # Start Streamlit
# def run():
#     subprocess.Popen(
#         ['python', '-m', 'streamlit', 'run', 'app.py',
#          '--server.port=8502',
#          '--server.headless=true',
#          '--server.enableCORS=false'],
#         stdout=subprocess.DEVNULL,
#         stderr=subprocess.DEVNULL
#     )

# threading.Thread(target=run, daemon=True).start()
# print('Waiting 60 seconds for app to start...')
# time.sleep(60)

# tunnel = ngrok.connect(8502)
# print(f'\n YOUR APP IS LIVE!')
# print(f'OPEN THIS URL: {tunnel.public_url}')
import subprocess, threading, time, os
from pyngrok import ngrok

os.system("pkill -f streamlit")
ngrok.kill()
time.sleep(5)

# Check files exist
files = os.listdir('.')
print('Files found:', files)

if 'app.py' not in files:
    print('app.py is MISSING - need to recreate it!')
elif 'energy_data.csv' not in files:
    print('energy_data.csv is MISSING - need to regenerate dataset!')
else:
    print('All files found - starting app...')

    # Start Streamlit
    proc = subprocess.Popen(
        ['python', '-m', 'streamlit', 'run', 'app.py',
         '--server.port=8502', '--server.headless=true',
         '--server.enableCORS=false'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    # Wait for startup
    print('Waiting 60 seconds...')
    time.sleep(60)

    # Check if running
    import urllib.request
    try:
        urllib.request.urlopen('http://localhost:8502')
        print('Streamlit is running!')
        ngrok.set_auth_token("3Fa5vHo8XqriknbjjVZu0XKpsHz_4aEPpnZu56jrL8EtYWbuQ")
        tunnel = ngrok.connect(8502)
        print(f'APP IS LIVE: {tunnel.public_url}')
    except Exception as e:
        print(f'Streamlit failed: {e}')
        # Print error logs
        err = proc.stderr.read(2000).decode()
        print('Error logs:', err)

Files found: ['.config', 'energy_data.csv', 'sample_data']
app.py is MISSING - need to recreate it!
